# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Section 1 checks (GUIDE step 1): counts, session leakage, boxes per class, visualize 6 train images
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

DATA = Path('data')
def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

splits = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in ('train', 'val', 'test')}
meta = json.load(open(DATA / 'splits' / 'meta.json', encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])

# 1) images / boxes per split
for s, rows in splits.items():
    boxes = Counter(o['label'] for r in rows for o in r['objects'])
    sizes = Counter((r['width'], r['height']) for r in rows)
    n_empty = sum(1 for r in rows if not r['objects'])
    n_sess = len({r['session'] for r in rows})
    print(f"{s:5s} images={len(rows):4d} sessions={n_sess:3d} empty_images={n_empty:3d} "
          f"boxes={dict(boxes)} total_boxes={sum(boxes.values())} sizes={dict(sizes)}")
    assert len(rows) == meta['counts'][s]['images']

# 2) no session (and no image) shared between splits
sess = {s: {r['session'] for r in rows} for s, rows in splits.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    inter = sess[a] & sess[b]
    print(f'sessions {a} & {b}: {len(inter)}')
    assert not inter, inter
ids = [r['image_id'] for rows in splits.values() for r in rows]
assert len(ids) == len(set(ids)), 'duplicate image_id across splits'
print('OK: no session / image_id shared between splits')

# 3) box size: median box area as % of frame (small boxes are hard)
areas = sorted(o['yolo'][2] * o['yolo'][3] for r in splits['train'] for o in r['objects'])
print(f"train median box area = {100 * areas[len(areas) // 2]:.2f}% of frame")

# 4) draw 6 train images with GT boxes (YOLO xc,yc,w,h normalized -> pixel corners)
COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
random.seed(0)
sample = random.sample([r for r in splits['train'] if r['objects']], 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, sample):
    img = Image.open(DATA / r['image_path']).convert('RGB')
    W, H = img.size
    d = ImageDraw.Draw(img)
    for o in r['objects']:
        xc, yc, w, h = o['yolo']
        box = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 3, box[1] + 3), o['label'], fill=COLORS[o['label']])
    ax.imshow(img)
    ax.set_title(f"{r['image_id']} | {r['session']} | {len(r['objects'])} boxes", fontsize=9)
    ax.axis('off')
plt.tight_layout()
Path('report/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('report/figures/step1_train_samples.png', dpi=80)
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Section 2 (GUIDE step 2): YOLO labels -> JSON answer on the 0-1000 scale + prompt
import json
from scripts.score import parse_output, iou_xyxy   # the referee's own parser / IoU (read-only use)

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo                                   # normalized [0,1], center + size
    x1, y1, x2, y2 = (min(1000, max(0, round(v * 1000)))
                      for v in (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2))
    # keep tiny boxes valid: score.parse_output rejects the WHOLE answer if any x2<=x1 or y2<=y1
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < 1000 else (999, 1000)
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < 1000 else (999, 1000)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))   # top->bottom, left->right: stable order
    return json.dumps(boxes, separators=(',', ':'))                 # "[]" if no object

PROMPT = (
    "Detect all road damage in this dashcam image. Classes: pothole, crack, manhole.\n"
    "Return ONLY a JSON list, with no extra text and no markdown, in the form "
    '[{"label": "pothole|crack|manhole", "bbox_2d": [x1, y1, x2, y2]}].\n'
    "bbox_2d coordinates are integers on a 0-1000 scale relative to the image width and height "
    "(x1,y1 = top-left corner, x2,y2 = bottom-right corner). If there is no damage, return []."
)

# Check 1: every answer in every split parses with the referee's parser, keeps all boxes,
# and the integer 0-1000 box stays close to the exact GT box the referee uses
worst_iou, worst_px, n_boxes, lens = 1.0, 0.0, 0, []
for s, rows in splits.items():
    for r in rows:
        ans = objects_to_answer(r['objects'])
        lens.append(len(ans))
        parsed = parse_output(ans)
        assert parsed is not None and len(parsed) == len(r['objects']), (s, r['image_id'], ans)
        for o in r['objects']:
            xc, yc, w, h = o['yolo']
            exact = ((xc - w / 2) * 1000, (yc - h / 2) * 1000, (xc + w / 2) * 1000, (yc + h / 2) * 1000)
            got = yolo_to_box(o['label'], o['yolo'])['bbox_2d']
            worst_iou = min(worst_iou, iou_xyxy(exact, tuple(got)))
            scale = (r['width'], r['height'], r['width'], r['height'])
            worst_px = max(worst_px, max(abs(g - e) * k / 1000 for g, e, k in zip(got, exact, scale)))
            n_boxes += 1
print(f"{n_boxes} boxes in {len(lens)} answers all parse OK with score.parse_output")
print(f"worst IoU(int 0-1000 box, exact GT) = {worst_iou:.3f}; worst corner error = {worst_px:.2f} px")
print(f"answer length: max={max(lens)} chars, mean={sum(lens) / len(lens):.0f} chars")

# Check 2: round trip on the busiest train image: JSON 0-1000 -> pixel must sit on the original YOLO box
r = max(splits['train'], key=lambda r: len(r['objects']))
ans = objects_to_answer(r['objects'])
print('\nPROMPT:\n' + PROMPT)
print(f"\nanswer for {r['image_id']} ({len(r['objects'])} boxes):\n{ans}")
img = Image.open(DATA / r['image_path']).convert('RGB')
W, H = img.size
d = ImageDraw.Draw(img)
for o in r['objects']:                          # original YOLO -> pixel (thick green)
    xc, yc, w, h = o['yolo']
    d.rectangle([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], outline='lime', width=5)
for b in json.loads(ans):                       # JSON 0-1000 -> pixel (thin red), should lie on the green box
    x1, y1, x2, y2 = b['bbox_2d']
    d.rectangle([x1 * W / 1000, y1 * H / 1000, x2 * W / 1000, y2 * H / 1000], outline='red', width=1)
    d.text((x1 * W / 1000 + 3, y1 * H / 1000 + 3), b['label'], fill='red')
plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.axis('off')
plt.title(f"Round trip {r['image_id']}: green = original YOLO box, red = JSON 0-1000 -> pixel")
plt.savefig('report/figures/step2_roundtrip.png', dpi=80, bbox_inches='tight')
plt.show()

In [ ]:
# Diagnose GT boxes whose integer 0-1000 version differs from the exact box the referee uses
# Found: 1 zero-height crack box (train, vlcsnap-00058) and 45 boxes extending outside the frame
bad = []
for s, rows in splits.items():
    for r in rows:
        for o in r['objects']:
            xc, yc, w, h = o['yolo']
            exact = ((xc - w / 2) * 1000, (yc - h / 2) * 1000, (xc + w / 2) * 1000, (yc + h / 2) * 1000)
            got = tuple(yolo_to_box(o['label'], o['yolo'])['bbox_2d'])
            v = iou_xyxy(exact, got)
            if v < 0.9:
                bad.append((round(v, 3), s, r['image_id'], o['label'], [round(x, 4) for x in o['yolo']],
                            [round(x, 1) for x in exact], list(got)))
print(len(bad), 'boxes with IoU(int, exact) < 0.9')
for b in sorted(bad)[:20]:
    print(b)
out_of_frame = sum(1 for rows in splits.values() for r in rows for o in r['objects']
                   if min(o['yolo'][0] - o['yolo'][2] / 2, o['yolo'][1] - o['yolo'][3] / 2) < -1e-6
                   or max(o['yolo'][0] + o['yolo'][2] / 2, o['yolo'][1] + o['yolo'][3] / 2) > 1 + 1e-6)
zero_size = sum(1 for rows in splits.values() for r in rows for o in r['objects'] if o['yolo'][2] <= 0 or o['yolo'][3] <= 0)
print('boxes extending outside the frame:', out_of_frame, '| zero-size boxes:', zero_size)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9